# Local Evidence Classification

Evaluate token-window evidence without leaking parent texts across partitions. MG is the positive class (`1`).

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from IPython.display import display
import pandas as pd

from classification_utils import plot_score_curves
from local_evidence_utils import (
    LOCAL_FEATURE_COLUMNS,
    build_local_evidence_windows,
    default_local_evidence_path,
    fit_local_evidence_model,
    load_or_build_local_evidence_windows,
    plot_local_evidence_timeline,
    plot_normalized_token_metrics_with_local_evidence,
    run_local_evidence_experiments,
    score_local_evidence_windows,
    split_local_evidence_texts,
)
from token_candidate_similarity_heatmaps_utils import (
    load_similarity_row,
    load_text_row,
    text_index_for_id,
    validated_text_summary,
    visualize_text_similarities,
)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
DATASET = "kaggle_train"
MODEL = "opt_350m"
WINDOW_LENGTHS = (32, 256)

STRIDE = 16
TOP_K = 20
SMOOTHING_WINDOW = 10
SEED = 42
RECOMPUTE_WINDOWS = False

pd.set_option("display.max_columns", 100)

In [ ]:
WINDOWS_PATH = default_local_evidence_path(PROCESSED_ROOT, DATASET, MODEL)
windows = load_or_build_local_evidence_windows(
        PROCESSED_ROOT,
        DATASET,
        MODEL,
        path=WINDOWS_PATH,
        window_lengths=WINDOW_LENGTHS,
        stride=STRIDE,
        top_k=TOP_K,
        recompute=RECOMPUTE_WINDOWS,
    )

display(windows.groupby(["split", "window_length"]).size().rename("window_rows").reset_index())
display(
    windows.loc[:, LOCAL_FEATURE_COLUMNS]
    .notna()
    .sum()
    .rename("non_null_windows")
    .rename_axis("feature")
    .reset_index()
)

In [ ]:
partitions = split_local_evidence_texts(windows, seed=SEED)
test_summary, test_predictions, scored_windows = run_local_evidence_experiments(
    windows,
    partitions,
    feature_columns=LOCAL_FEATURE_COLUMNS,
    seed=SEED,
)
display(test_summary.sort_values(["window_length", "aggregation"]).reset_index(drop=True))
curve_predictions = test_predictions.assign(
    mg_score=lambda frame: frame["mg_score"].where(
        frame["score_direction"].eq("higher_is_mg"),
        -frame["mg_score"],
    ),
        experiment=lambda frame: frame["window_length"].astype(str) + " tokens: " + frame["aggregation"]
)
plot_score_curves(curve_predictions)

In [ ]:
prediction_scores = test_predictions.assign(
    decision_score=lambda frame: frame["mg_score"].where(
        frame["score_direction"].eq("higher_is_mg"),
        -frame["mg_score"],
    ),
    decision_margin=lambda frame: (frame["decision_score"] - frame["document_threshold"]).abs(),
).sort_values("decision_score", ascending=False).reset_index(drop=True)
MOST_SUSPICIOUS = prediction_scores.head(1)
MOST_CONFIDENT_ERROR = prediction_scores.loc[
    prediction_scores["prediction"].ne(prediction_scores["label"])
].nlargest(1, "decision_margin")

In [ ]:
INSPECT_TEXT = None
INSPECT_SPLIT = "MG"
INSPECT_WINDOW_LENGTH = 256
INSPECT_AGGREGATION = "maximum_window"

manual_selection = prediction_scores.iloc[0:0]
if INSPECT_TEXT is not None:
    manual_selection = prediction_scores.loc[
        prediction_scores["split"].eq(INSPECT_SPLIT)
        & prediction_scores["text_id"].eq(INSPECT_TEXT)
        & prediction_scores["window_length"].eq(INSPECT_WINDOW_LENGTH)
        & prediction_scores["aggregation"].eq(INSPECT_AGGREGATION)
    ]

selections = pd.concat([MOST_SUSPICIOUS, MOST_CONFIDENT_ERROR, manual_selection]).drop_duplicates(
    ["split", "text_id", "window_length", "aggregation"]
)
for _, prediction in selections.iterrows():
    test_windows = scored_windows.loc[
        scored_windows["partition"].eq("test")
        & scored_windows["split"].eq(prediction["split"])
        & scored_windows["text_id"].eq(prediction["text_id"])
        & scored_windows["window_length"].eq(prediction["window_length"])
    ].assign(window_threshold=prediction["window_threshold"])
    display(plot_local_evidence_timeline(test_windows, prediction))
    visualize_text_similarities(
        PROCESSED_ROOT,
        DATASET,
        MODEL,
        prediction["split"],
        text_index_for_id(PROCESSED_ROOT, DATASET, MODEL, prediction["split"], prediction["text_id"]),
        top_k=TOP_K,
        token_range=(prediction["selected_start_token"], prediction["selected_stop_token"]),
    )

In [ ]:
CUSTOM_TEXT_INDEX = 0
CUSTOM_SPLIT = "MG"
CUSTOM_WINDOW_LENGTH = 32

custom_row = load_text_row(PROCESSED_ROOT, DATASET, MODEL, CUSTOM_SPLIT, CUSTOM_TEXT_INDEX)
custom_text_id = int(custom_row["id"])
custom_windows = build_local_evidence_windows(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    window_lengths=(CUSTOM_WINDOW_LENGTH,),
    stride=STRIDE,
    top_k=TOP_K,
    splits=(CUSTOM_SPLIT,),
    text_ids=(custom_text_id,),
    show_progress=False,
).loc[lambda frame: frame["text_id"].eq(custom_text_id)].copy()
if custom_windows.empty:
    raise ValueError(
        f"No windows available for custom text id={custom_text_id} "
        f"(index={CUSTOM_TEXT_INDEX}, split={CUSTOM_SPLIT}, window_length={CUSTOM_WINDOW_LENGTH})"
    )

custom_model, custom_features = fit_local_evidence_model(
    windows,
    partitions,
    CUSTOM_WINDOW_LENGTH,
    LOCAL_FEATURE_COLUMNS,
    SEED,
)
custom_windows = score_local_evidence_windows(custom_model, custom_windows, custom_features)
custom_similarity_row = load_similarity_row(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    CUSTOM_SPLIT,
    custom_text_id,
)
custom_summary, custom_token_labels = validated_text_summary(custom_row, custom_similarity_row, top_k=TOP_K)
# validated_text_summary(...) builds the token_summary_series(...) used below.
plot_normalized_token_metrics_with_local_evidence(
    custom_summary,
    custom_windows,
    len(custom_token_labels),
    window=SMOOTHING_WINDOW,
)
custom_prediction = custom_windows.loc[custom_windows["mg_score"].idxmax()].copy()
custom_prediction["selected_start_token"] = custom_prediction["start_token"]
custom_prediction["selected_stop_token"] = custom_prediction["stop_token"]
custom_prediction["document_threshold"] = float("nan")
custom_prediction["prediction"] = "maximum_window"
custom_windows["window_threshold"] = float("nan")
display("Independent custom inspection evidence")
display(plot_local_evidence_timeline(custom_windows, custom_prediction))
visualize_text_similarities(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    CUSTOM_SPLIT,
    CUSTOM_TEXT_INDEX,
    top_k=TOP_K,
    token_range=(custom_prediction["selected_start_token"], custom_prediction["selected_stop_token"]),
)

In [ ]:
CUSTOM_TEXT_INDEX = 0
CUSTOM_SPLIT = "HG"
CUSTOM_WINDOW_LENGTH = 32

custom_row = load_text_row(PROCESSED_ROOT, DATASET, MODEL, CUSTOM_SPLIT, CUSTOM_TEXT_INDEX)
custom_text_id = int(custom_row["id"])
custom_windows = build_local_evidence_windows(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    window_lengths=(CUSTOM_WINDOW_LENGTH,),
    stride=STRIDE,
    top_k=TOP_K,
    splits=(CUSTOM_SPLIT,),
    text_ids=(custom_text_id,),
    show_progress=False,
).loc[lambda frame: frame["text_id"].eq(custom_text_id)].copy()
if custom_windows.empty:
    raise ValueError(
        f"No windows available for custom text id={custom_text_id} "
        f"(index={CUSTOM_TEXT_INDEX}, split={CUSTOM_SPLIT}, window_length={CUSTOM_WINDOW_LENGTH})"
    )

custom_model, custom_features = fit_local_evidence_model(
    windows,
    partitions,
    CUSTOM_WINDOW_LENGTH,
    LOCAL_FEATURE_COLUMNS,
    SEED,
)
custom_windows = score_local_evidence_windows(custom_model, custom_windows, custom_features)
custom_similarity_row = load_similarity_row(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    CUSTOM_SPLIT,
    custom_text_id,
)
custom_summary, custom_token_labels = validated_text_summary(custom_row, custom_similarity_row, top_k=TOP_K)
# validated_text_summary(...) builds the token_summary_series(...) used below.
plot_normalized_token_metrics_with_local_evidence(
    custom_summary,
    custom_windows,
    len(custom_token_labels),
    window=SMOOTHING_WINDOW,
)
custom_prediction = custom_windows.loc[custom_windows["mg_score"].idxmax()].copy()
custom_prediction["selected_start_token"] = custom_prediction["start_token"]
custom_prediction["selected_stop_token"] = custom_prediction["stop_token"]
custom_prediction["document_threshold"] = float("nan")
custom_prediction["prediction"] = "maximum_window"
custom_windows["window_threshold"] = float("nan")
display("Independent custom inspection evidence")
display(plot_local_evidence_timeline(custom_windows, custom_prediction))
visualize_text_similarities(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    CUSTOM_SPLIT,
    CUSTOM_TEXT_INDEX,
    top_k=TOP_K,
    token_range=(custom_prediction["selected_start_token"], custom_prediction["selected_stop_token"]),
)

In [ ]:
CUSTOM_TEXT_INDEX = 10
CUSTOM_SPLIT = "HG"
CUSTOM_WINDOW_LENGTH = 32

custom_row = load_text_row(PROCESSED_ROOT, DATASET, MODEL, CUSTOM_SPLIT, CUSTOM_TEXT_INDEX)
custom_text_id = int(custom_row["id"])
custom_windows = build_local_evidence_windows(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    window_lengths=(CUSTOM_WINDOW_LENGTH,),
    stride=STRIDE,
    top_k=TOP_K,
    splits=(CUSTOM_SPLIT,),
    text_ids=(custom_text_id,),
    show_progress=False,
).loc[lambda frame: frame["text_id"].eq(custom_text_id)].copy()
if custom_windows.empty:
    raise ValueError(
        f"No windows available for custom text id={custom_text_id} "
        f"(index={CUSTOM_TEXT_INDEX}, split={CUSTOM_SPLIT}, window_length={CUSTOM_WINDOW_LENGTH})"
    )

custom_model, custom_features = fit_local_evidence_model(
    windows,
    partitions,
    CUSTOM_WINDOW_LENGTH,
    LOCAL_FEATURE_COLUMNS,
    SEED,
)
custom_windows = score_local_evidence_windows(custom_model, custom_windows, custom_features)
custom_similarity_row = load_similarity_row(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    CUSTOM_SPLIT,
    custom_text_id,
)
custom_summary, custom_token_labels = validated_text_summary(custom_row, custom_similarity_row, top_k=TOP_K)
# validated_text_summary(...) builds the token_summary_series(...) used below.
plot_normalized_token_metrics_with_local_evidence(
    custom_summary,
    custom_windows,
    len(custom_token_labels),
    window=SMOOTHING_WINDOW,
)
custom_prediction = custom_windows.loc[custom_windows["mg_score"].idxmax()].copy()
custom_prediction["selected_start_token"] = custom_prediction["start_token"]
custom_prediction["selected_stop_token"] = custom_prediction["stop_token"]
custom_prediction["document_threshold"] = float("nan")
custom_prediction["prediction"] = "maximum_window"
custom_windows["window_threshold"] = float("nan")
display("Independent custom inspection evidence")
display(plot_local_evidence_timeline(custom_windows, custom_prediction))
visualize_text_similarities(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    CUSTOM_SPLIT,
    CUSTOM_TEXT_INDEX,
    top_k=TOP_K,
    token_range=(custom_prediction["selected_start_token"], custom_prediction["selected_stop_token"]),
)

In [ ]:
CUSTOM_TEXT_INDEX = 10
CUSTOM_SPLIT = "MG"
CUSTOM_WINDOW_LENGTH = 32

custom_row = load_text_row(PROCESSED_ROOT, DATASET, MODEL, CUSTOM_SPLIT, CUSTOM_TEXT_INDEX)
custom_text_id = int(custom_row["id"])
custom_windows = build_local_evidence_windows(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    window_lengths=(CUSTOM_WINDOW_LENGTH,),
    stride=STRIDE,
    top_k=TOP_K,
    splits=(CUSTOM_SPLIT,),
    text_ids=(custom_text_id,),
    show_progress=False,
).loc[lambda frame: frame["text_id"].eq(custom_text_id)].copy()
if custom_windows.empty:
    raise ValueError(
        f"No windows available for custom text id={custom_text_id} "
        f"(index={CUSTOM_TEXT_INDEX}, split={CUSTOM_SPLIT}, window_length={CUSTOM_WINDOW_LENGTH})"
    )

custom_model, custom_features = fit_local_evidence_model(
    windows,
    partitions,
    CUSTOM_WINDOW_LENGTH,
    LOCAL_FEATURE_COLUMNS,
    SEED,
)
custom_windows = score_local_evidence_windows(custom_model, custom_windows, custom_features)
custom_similarity_row = load_similarity_row(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    CUSTOM_SPLIT,
    custom_text_id,
)
custom_summary, custom_token_labels = validated_text_summary(custom_row, custom_similarity_row, top_k=TOP_K)
# validated_text_summary(...) builds the token_summary_series(...) used below.
plot_normalized_token_metrics_with_local_evidence(
    custom_summary,
    custom_windows,
    len(custom_token_labels),
    window=SMOOTHING_WINDOW,
)
custom_prediction = custom_windows.loc[custom_windows["mg_score"].idxmax()].copy()
custom_prediction["selected_start_token"] = custom_prediction["start_token"]
custom_prediction["selected_stop_token"] = custom_prediction["stop_token"]
custom_prediction["document_threshold"] = float("nan")
custom_prediction["prediction"] = "maximum_window"
custom_windows["window_threshold"] = float("nan")
display("Independent custom inspection evidence")
display(plot_local_evidence_timeline(custom_windows, custom_prediction))
visualize_text_similarities(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    CUSTOM_SPLIT,
    CUSTOM_TEXT_INDEX,
    top_k=TOP_K,
    token_range=(custom_prediction["selected_start_token"], custom_prediction["selected_stop_token"]),
)